# Hate Speech Classification with Robustness Testing
**Dataset:** HateXplain (Mathew et al., 2021)  
**Model:** HateBERT fine-tuned for 3-class classification (hate / offensive / normal)  
**Focus:** Robustness to text obfuscation — leet-speak, punctuation insertion, character repetition

---

## ⚙️ Configuration

All hyperparameters and paths are set here. Edit this cell to change the experiment.

In [ ]:
# ── Install dependencies (run once, then Runtime → Restart session) ──
# !pip install transformers torch scikit-learn matplotlib seaborn wordcloud nltk --quiet

# ── Configuration — edit here ────────────────────────────
MODEL_NAME   = "GroNLP/hateBERT"   # change to "distilbert-base-uncased" to use DistilBERT
MAX_LEN      = 128                  # max token length for tokenizer
BATCH_SIZE   = 16                   # reduce to 8 if OOM on Colab free tier
LR           = 2e-5                 # learning rate
WEIGHT_DECAY = 0.1                  # L2 regularization
MAX_EPOCHS   = 8                    # upper bound — early stopping will cut this short
PATIENCE     = 2                    # early stopping patience (epochs without val loss improvement)
AUG_RATE     = 0.5                  # fraction of hate/offensive posts augmented per epoch (Task 4c)
RANDOM_SEED  = 42

# ── Paths ─────────────────────────────────────────────────
DRIVE_DIR    = "/content/drive/MyDrive/hate_speech_project"  # where to save models
MODEL_SAVE   = "best_model_baseline"
MODEL_IMP_SAVE = "best_model_improved"

# ── Label mapping ─────────────────────────────────────────
LABEL_MAP    = {"hatespeech": 0, "offensive": 1, "normal": 2}
LABEL_NAMES  = {0: "hate", 1: "offensive", 2: "normal"}
COLORS       = {"hate": "#e63946", "offensive": "#f4a261", "normal": "#2a9d8f"}

## 📦 Imports & Shared Utilities

All imports and shared functions (dataset class, training loop, obfuscation) are defined once here and reused across tasks.

In [ ]:
import json, re, random, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch.nn as nn

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))
random.seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device}")
if device.type == "cuda":
    print(f"  GPU   : {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Shared: PyTorch Dataset ───────────────────────────────
class HateDataset(Dataset):
    """Tokenizes text on-the-fly. Optionally applies augmentation to hate/offensive posts."""
    def __init__(self, texts, labels, tokenizer, augment_fn=None, aug_rate=0.5):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.augment_fn = augment_fn
        self.aug_rate   = aug_rate

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text  = self.texts[idx]
        label = self.labels[idx]
        if self.augment_fn and label in [0, 1] and random.random() < self.aug_rate:
            text = self.augment_fn(text)
        enc = self.tokenizer(text, max_length=MAX_LEN, padding='max_length',
                             truncation=True, return_tensors='pt')
        return {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label':          torch.tensor(label, dtype=torch.long)}

In [ ]:
# ── Shared: evaluate() ────────────────────────────────────
def evaluate(model, loader, criterion=None):
    """Returns macro F1, avg loss, predictions, labels."""
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0
    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(device)
            mask  = batch["attention_mask"].to(device)
            lbls  = batch["label"].to(device)
            out   = model(input_ids=ids, attention_mask=mask)
            loss  = (criterion(out.logits, lbls) if criterion
                     else model(input_ids=ids, attention_mask=mask, labels=lbls).loss)
            total_loss += loss.item()
            all_preds.extend(out.logits.argmax(dim=-1).cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
    return (f1_score(all_labels, all_preds, average="macro"),
            total_loss / len(loader), all_preds, all_labels)


# ── Shared: get_predictions() ─────────────────────────────
def get_predictions(model, tokenizer, texts, labels, batch_size=32):
    """Run inference on raw text lists, return preds and probs."""
    ds     = HateDataset(texts, labels, tokenizer)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    all_preds, all_probs = [], []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            out   = model(input_ids=batch["input_ids"].to(device),
                          attention_mask=batch["attention_mask"].to(device))
            probs = torch.softmax(out.logits, dim=-1)
            all_preds.extend(probs.argmax(dim=-1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_probs)

In [ ]:
# ── Shared: Obfuscation functions ────────────────────────
LEET_MAP = {'a':'4','e':'3','i':'1','o':'0','s':'5','t':'7','g':'9','b':'8'}

def leet_speak(text, rate=0.4):
    """Replace letters with leet equivalents (e.g. hate → h4t3)."""
    return "".join(LEET_MAP[c.lower()] if c.lower() in LEET_MAP
                   and random.random() < rate else c for c in text)

def insert_punctuation(text, rate=0.3):
    """Insert punctuation between chars of random words (e.g. hate → h.a.t.e)."""
    words = text.split()
    for i, w in enumerate(words):
        if len(w) > 3 and random.random() < rate:
            words[i] = random.choice([".","-","_","*"]).join(list(w))
    return " ".join(words)

def char_repeat(text, rate=0.3):
    """Randomly repeat a character in random words (e.g. hate → haate)."""
    words = text.split()
    for i, w in enumerate(words):
        if len(w) > 3 and random.random() < rate:
            idx = random.randint(1, len(w)-2)
            words[i] = w[:idx] + w[idx]*random.randint(2,4) + w[idx+1:]
    return " ".join(words)

def combined_obfuscation(text):
    """Apply all three obfuscation strategies."""
    return char_repeat(insert_punctuation(leet_speak(text, 0.3), 0.2), 0.2)

OBFUSCATION_FNS = {
    "leet_speak":  leet_speak,
    "punctuation": insert_punctuation,
    "char_repeat": char_repeat,
    "combined":    combined_obfuscation,
}

# Preview
random.seed(RANDOM_SEED)
s = "I hate those people they should all leave"
print(f"Original    : {s}")
print(f"Leet        : {leet_speak(s)}")
print(f"Punctuation : {insert_punctuation(s)}")
print(f"Char repeat : {char_repeat(s)}")
print(f"Combined    : {combined_obfuscation(s)}")

In [ ]:
# ── Shared: training loop with early stopping ─────────────
def train_model(model, train_loader, val_loader, epochs, lr,
                weight_decay, patience, criterion=None, label=""):
    """
    Fine-tune model with AdamW + linear warmup + early stopping on val loss.
    Returns best model state dict and training history.
    """
    n_steps  = len(train_loader) * epochs
    n_warmup = int(0.1 * n_steps)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = get_scheduler("linear", optimizer=optimizer,
                               num_warmup_steps=n_warmup, num_training_steps=n_steps)

    history       = {"train_loss": [], "val_loss": [], "val_f1": []}
    best_val_loss = float("inf")
    best_state    = None
    no_improve    = 0
    stopped_at    = epochs

    print(f"Training {label} | max {epochs} epochs | patience {patience}\n")
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(train_loader):
            ids, mask, lbls = (batch["input_ids"].to(device),
                               batch["attention_mask"].to(device),
                               batch["label"].to(device))
            optimizer.zero_grad()
            out  = model(input_ids=ids, attention_mask=mask)
            loss = criterion(out.logits, lbls) if criterion else \
                   model(input_ids=ids, attention_mask=mask, labels=lbls).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()
            if (step+1) % 100 == 0:
                print(f"  Ep {epoch+1} | Step {step+1}/{len(train_loader)} | Loss {loss.item():.4f}")

        avg_train = total_loss / len(train_loader)
        val_f1, avg_val, _, _ = evaluate(model, val_loader)
        history["train_loss"].append(avg_train)
        history["val_loss"].append(avg_val)
        history["val_f1"].append(val_f1)

        print(f"\n── Epoch {epoch+1}/{epochs} ──")
        print(f"   Train loss : {avg_train:.4f}")
        print(f"   Val loss   : {avg_val:.4f}")
        print(f"   Val F1     : {val_f1:.4f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve    = 0
            print(f"   ✓ Best model (val_loss={best_val_loss:.4f}, F1={val_f1:.4f})")
        else:
            no_improve += 1
            print(f"   ✗ No improvement ({no_improve}/{patience})")
            if no_improve >= patience:
                stopped_at = epoch + 1
                print(f"\n⚡ Early stopping at epoch {stopped_at}")
                break
        print()

    print(f"Done. Best val_loss={best_val_loss:.4f}")
    return best_state, history, stopped_at
# ── Shared: plot training curves ─────────────────────────
def plot_curves(history, stopped_at=None, title="", save_path="curves.png"):
    n = len(history["train_loss"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(range(1,n+1), history["train_loss"], marker="o", label="Train", color="#457b9d")
    axes[0].plot(range(1,n+1), history["val_loss"],   marker="o", label="Val",   color="#e63946")
    if stopped_at and stopped_at < MAX_EPOCHS:
        axes[0].axvline(stopped_at - PATIENCE, color="gray", linestyle="--", label="Early stop")
    axes[0].set_title(f"Loss per Epoch {title}", fontweight="bold")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].spines[["top","right"]].set_visible(False)
    axes[1].plot(range(1,n+1), history["val_f1"], marker="o", color="#2a9d8f")
    axes[1].set_title(f"Val Macro F1 {title}", fontweight="bold")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro F1")
    axes[1].spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"✓ Saved {save_path}")


# ── Shared: confusion matrix plot ────────────────────────
def plot_cm(labels, preds, title="", cmap="Blues", save_path="cm.png"):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=["hate","offensive","normal"],
                yticklabels=["hate","offensive","normal"], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix {title}", fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"✓ Saved {save_path}")


# ── Shared: robustness evaluation ────────────────────────
def robustness_eval(model, tokenizer, texts, labels, reference_f1=None, obf_fns=None):
    """Evaluate model on clean + obfuscated variants. Returns results dict."""
    if obf_fns is None:
        obf_fns = OBFUSCATION_FNS
    random.seed(RANDOM_SEED)
    results, obf_cache = {}, {"clean": texts}
    preds_clean, _ = get_predictions(model, tokenizer, texts, labels)
    results["clean"] = f1_score(labels, preds_clean, average="macro")
    ref = reference_f1 or results["clean"]
    print(f"  clean          → F1: {results['clean']:.4f}")
    for name, fn in obf_fns.items():
        obf = [fn(t) for t in texts]
        obf_cache[name] = obf
        p, _ = get_predictions(model, tokenizer, obf, labels)
        f1   = f1_score(labels, p, average="macro")
        results[name] = f1
        print(f"  {name:<16} → F1: {f1:.4f}  (drop: {ref-f1:+.4f})")
    return results, obf_cache, preds_clean

print("✓ All shared utilities defined")

---
## Task 1 — Problem Definition & Dataset Understanding

**Task:** Classify social media posts into three categories — *hate speech*, *offensive language*, and *normal*.  
This is an NLP classification problem because it requires understanding context, implicit meaning, and social norms from raw text — not just surface-level keywords.

**Dataset:** [HateXplain](https://github.com/hate-alert/HateXplain)

**Known limitations:** class imbalance; annotation subjectivity; English-only; no demographic information on annotators.

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/"

def download_json(filename):
    with urllib.request.urlopen(BASE_URL + filename) as r:
        return json.loads(r.read().decode())

print("Downloading dataset from GitHub...")
data       = download_json("dataset.json")
split_info = download_json("post_id_divisions.json")
print(f"✓ Total posts: {len(data)} | Splits: {list(split_info.keys())}")

# Inspect one example
sample_id = list(data.keys())[0]
print(f"\n── Sample post ({sample_id}) ──")
print(json.dumps(data[sample_id], indent=2)[:600])

In [ ]:
def majority_vote(label_list):
    count = Counter(label_list)
    top   = count.most_common(1)[0]
    return top[0] if top[1] >= 2 else None

def build_dataframe(post_ids):
    rows = []
    for pid in post_ids:
        if pid not in data: continue
        post   = data[pid]
        labels = [a["label"] for a in post["annotators"]]
        vote   = majority_vote(labels)
        if vote is None: continue
        rows.append({"post_id": pid,
                     "text":       " ".join(post["post_tokens"]),
                     "label":      LABEL_MAP[vote],
                     "label_name": LABEL_NAMES[LABEL_MAP[vote]]})
    return pd.DataFrame(rows)

train_df = build_dataframe(split_info["train"])
val_df   = build_dataframe(split_info["val"])
test_df  = build_dataframe(split_info["test"])

print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")
print(f"\nLabel distribution (train):\n{train_df['label_name'].value_counts()}")

train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv",     index=False)
test_df.to_csv("test.csv",   index=False)
print("\n✓ Saved train.csv / val.csv / test.csv")

---
## Task 2 — Exploratory Data Analysis

In [ ]:
train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("val.csv")
test_df  = pd.read_csv("test.csv")

# ── Class distribution ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, df) in zip(axes, [("Train",train_df),("Validation",val_df),("Test",test_df)]):
    counts = df["label_name"].value_counts().reindex(["hate","offensive","normal"])
    bars   = ax.bar(counts.index, counts.values,
                    color=[COLORS[l] for l in counts.index], edgecolor="white")
    ax.set_title(name, fontsize=13, fontweight="bold")
    ax.set_xlabel("Class"); ax.set_ylabel("Count")
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                str(v), ha="center", fontsize=10)
    ax.spines[["top","right"]].set_visible(False)
plt.suptitle("Class Distribution across Splits", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("eda_class_distribution.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved eda_class_distribution.png")

In [ ]:
# ── Text length analysis ──────────────────────────────────
train_df["token_count"] = train_df["text"].apply(lambda x: len(str(x).split()))
train_df["char_count"]  = train_df["text"].apply(lambda x: len(str(x)))

print("── Token count by class ──")
print(train_df.groupby("label_name")["token_count"].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for label, color in COLORS.items():
    axes[0].hist(train_df[train_df["label_name"]==label]["token_count"],
                 bins=40, alpha=0.6, label=label, color=color, edgecolor="none")
axes[0].set_title("Token Count Distribution", fontweight="bold")
axes[0].set_xlabel("Tokens"); axes[0].set_ylabel("Frequency"); axes[0].legend()
axes[0].spines[["top","right"]].set_visible(False)

sns.boxplot(data=train_df, x="label_name", y="token_count", palette=COLORS,
            order=["hate","offensive","normal"], ax=axes[1], hue="label_name")
axes[1].set_title("Token Count Boxplot", fontweight="bold")
axes[1].spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("eda_text_length.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved eda_text_length.png")

In [ ]:
# ── Top-20 tokens per class ───────────────────────────────
def get_top_tokens(texts, n=20):
    tokens = []
    for text in texts:
        tokens.extend([w.lower() for w in str(text).split()
                       if w.isalpha() and w.lower() not in STOPWORDS and len(w) > 2])
    return Counter(tokens).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (label, color) in zip(axes, COLORS.items()):
    top = get_top_tokens(train_df[train_df["label_name"]==label]["text"])
    words, counts = zip(*top)
    ax.barh(words[::-1], counts[::-1], color=color, edgecolor="none")
    ax.set_title(f"Top 20 Tokens — {label.capitalize()}", fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)
plt.suptitle("Most Frequent Tokens by Class (NLTK stopwords removed)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_top_tokens.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved eda_top_tokens.png")

In [ ]:
# ── Word clouds ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (label, cmap) in zip(axes, {"hate":"Reds","offensive":"Oranges","normal":"Greens"}.items()):
    text  = " ".join(train_df[train_df["label_name"]==label]["text"].tolist())
    clean = " ".join([w for w in text.split()
                      if w.lower() not in STOPWORDS and w.isalpha() and len(w) > 2])
    wc = WordCloud(width=600, height=400, background_color="white",
                   colormap=cmap, max_words=100, collocations=False).generate(clean)
    ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
    ax.set_title(label.capitalize(), fontsize=13, fontweight="bold")
plt.suptitle("Word Clouds by Class", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_wordclouds.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved eda_wordclouds.png")

In [ ]:
# ── Summary statistics ────────────────────────────────────
print("══════════════════════════════════════════")
print("        EDA SUMMARY — TRAIN SPLIT")
print("══════════════════════════════════════════")
total = len(train_df)
for label in ["hate","offensive","normal"]:
    n   = (train_df["label_name"]==label).sum()
    avg = train_df[train_df["label_name"]==label]["token_count"].mean()
    print(f"  {label:<12} | n={n:>4} ({n/total*100:.1f}%) | avg tokens={avg:.1f}")
print(f"  {'TOTAL':<12} | n={total}")
print("══════════════════════════════════════════")
print(f"  Overall avg tokens : {train_df['token_count'].mean():.1f}")
print(f"  Overall avg chars  : {train_df['char_count'].mean():.1f}")
print(f"  Max token count    : {train_df['token_count'].max()}")
print(f"  Min token count    : {train_df['token_count'].min()}")

---
## Task 3 — Data Preprocessing

Preprocessing is intentionally **minimal**: HateBERT is pretrained on raw social media text, so aggressive normalization would hurt performance. We only replace URLs and @mentions with placeholder tokens, and strip `#` from hashtags.

HateBERT uses **WordPiece** subword tokenization — rare or long words are split into known subword pieces (e.g. `foreigners` → `foreign`, `##ers`). `max_length=128` covers >95% of posts in this dataset.

In [ ]:
train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("val.csv")
test_df  = pd.read_csv("test.csv")

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "[URL]",  text)
    text = re.sub(r"@\w+",            "[USER]", text)
    text = re.sub(r"#(\w+)",           r"\1",   text)
    text = re.sub(r"\s+",              " ",      text).strip()
    return text

for df in [train_df, val_df, test_df]:
    df["clean_text"] = df["text"].apply(clean_text)

print("── Before / After cleaning ──")
for i in range(2):
    print(f"\nOriginal : {train_df['text'].iloc[i]}")
    print(f"Cleaned  : {train_df['clean_text'].iloc[i]}")

In [ ]:
# ── Load tokenizer & check token lengths ─────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✓ Tokenizer: {MODEL_NAME} | Vocab size: {tokenizer.vocab_size}")

sample = train_df["clean_text"].iloc[0]
print(f"\nExample tokenization:")
print(f"  Text   : {sample}")
print(f"  Tokens : {tokenizer.tokenize(sample)[:20]}")

print("\nComputing token lengths...")
train_df["token_len"] = train_df["clean_text"].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True)))

plt.figure(figsize=(8,4))
plt.hist(train_df["token_len"], bins=50, color="#457b9d", edgecolor="none")
plt.axvline(MAX_LEN, color="red", linestyle="--", label=f"max_length={MAX_LEN}")
plt.title("Token Length Distribution (train)", fontweight="bold")
plt.xlabel("Token count"); plt.ylabel("Frequency"); plt.legend()
plt.tight_layout()
plt.savefig("preprocessing_token_lengths.png", dpi=150, bbox_inches="tight"); plt.show()

pct = (train_df["token_len"] <= MAX_LEN).mean() * 100
print(f"✓ Posts covered by max_length={MAX_LEN}: {pct:.1f}%")

In [ ]:
# ── Save clean CSVs ───────────────────────────────────────
cols = ["post_id","text","clean_text","label","label_name"]
train_df[cols].to_csv("train_clean.csv", index=False)
val_df[cols].to_csv("val_clean.csv",     index=False)
test_df[cols].to_csv("test_clean.csv",   index=False)
print("✓ Saved train_clean.csv / val_clean.csv / test_clean.csv")

---
## Task 4a — Baseline Training (HateBERT + Early Stopping)

**Model:** HateBERT (`GroNLP/hateBERT`) — BERT-base pretrained on ~1.5M Reddit posts from banned communities. Drop-in replacement for BERT-base with stronger prior knowledge of offensive language.

**Training setup:**
- AdamW optimizer, lr=2e-5, weight decay=0.1
- Linear scheduler with 10% warmup
- Early stopping on validation loss (patience=2)

In [ ]:
train_df = pd.read_csv("train_clean.csv")
val_df   = pd.read_csv("val_clean.csv")
test_df  = pd.read_csv("test_clean.csv")

# Datasets & loaders
train_ds_4a = HateDataset(train_df["clean_text"].tolist(), train_df["label"].tolist(), tokenizer)
val_ds_4a   = HateDataset(val_df["clean_text"].tolist(),   val_df["label"].tolist(),   tokenizer)
test_ds_4a  = HateDataset(test_df["clean_text"].tolist(),  test_df["label"].tolist(),  tokenizer)

train_loader_4a = DataLoader(train_ds_4a, batch_size=BATCH_SIZE, shuffle=True)
val_loader_4a   = DataLoader(val_ds_4a,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_4a  = DataLoader(test_ds_4a,  batch_size=BATCH_SIZE, shuffle=False)

print(f"✓ Train batches: {len(train_loader_4a)} | Val batches: {len(val_loader_4a)}")

# Model
model_4a = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
n_params = sum(p.numel() for p in model_4a.parameters() if p.requires_grad)
print(f"✓ Model: {MODEL_NAME} | Params: {n_params:,}")

In [ ]:
# ── Hate lexicon + targeted obfuscation ──────────────────
from collections import Counter

hate_counts = Counter()
all_counts  = Counter()
for text, label in zip(train_df['clean_text'], train_df['label']):
    words = str(text).lower().split()
    all_counts.update(words)
    if label in [0, 1]:
        hate_counts.update(words)

hate_lexicon = {
    w for w, c in hate_counts.items()
    if c >= 5 and c / (all_counts[w] + 1e-9) >= 0.70
}

def targeted_obfuscation(text):
    """Obfuscate only words in the hate lexicon."""
    words = text.split()
    return ' '.join(
        combined_obfuscation(w) if w.lower() in hate_lexicon else w
        for w in words
    )

print(f'✓ Hate lexicon: {len(hate_lexicon)} words')
print(f'  Sample: {list(hate_lexicon)[:10]}')

In [ ]:
# ── Train ─────────────────────────────────────────────────
best_state_4a, history_4a, stopped_4a = train_model(
    model_4a, train_loader_4a, val_loader_4a,
    epochs=MAX_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
    patience=PATIENCE, label="— HateBERT baseline"
)
plot_curves(history_4a, stopped_4a, title="— HateBERT Baseline",
            save_path="training_curves_4a.png")

In [ ]:
# ── Restore best model & evaluate on test set ─────────────
model_4a.load_state_dict(best_state_4a)
_, _, preds_4a_test, labels_4a_test = evaluate(model_4a, test_loader_4a)

print("── Task 4a — Test Set Results ──")
print(classification_report(labels_4a_test, preds_4a_test,
                             target_names=["hate","offensive","normal"], digits=4))
plot_cm(labels_4a_test, preds_4a_test,
        title="— HateBERT Baseline", cmap="Blues",
        save_path="cm_4a.png")

# Save
model_4a.save_pretrained(MODEL_SAVE)
tokenizer.save_pretrained(MODEL_SAVE)
print(f"✓ Saved to ./{MODEL_SAVE}/")

### Task 4a — Robustness (Targeted Obfuscation)

In [ ]:
# ── Baseline robustness: hate-lexicon words only ─────────
test_texts  = test_df['clean_text'].tolist()
test_labels = test_df['label'].tolist()

print('── Baseline robustness ──')
results_4a, obf_cache_4a, preds_clean_4a = robustness_eval(
    model_4a, tokenizer, test_texts, test_labels,
    obf_fns={'targeted': targeted_obfuscation})

clean_f1 = results_4a['clean']
obf_f1   = results_4a['targeted']
print(f'  clean    → F1: {clean_f1:.4f}')
print(f'  targeted → F1: {obf_f1:.4f}  (drop: {clean_f1-obf_f1:+.4f})')

p_obf, _ = get_predictions(model_4a, tokenizer, obf_cache_4a['targeted'], test_labels)
f1s_c = f1_score(test_labels, preds_clean_4a, average=None)
f1s_o = f1_score(test_labels, p_obf, average=None)
header = f"{'Condition':<12} {'hate':>8} {'offensive':>10} {'normal':>8} {'macro':>8}"
print('\n' + header); print('─'*len(header))
print(f'  clean        {f1s_c[0]:8.4f} {f1s_c[1]:10.4f} {f1s_c[2]:8.4f} {clean_f1:8.4f}')
print(f'  targeted     {f1s_o[0]:8.4f} {f1s_o[1]:10.4f} {f1s_o[2]:8.4f} {obf_f1:8.4f}')

random.seed(RANDOM_SEED)
failures = [
    {'text': test_texts[i][:80], 'obf': obf_cache_4a['targeted'][i][:80],
     'true': LABEL_NAMES[tl], 'clean_pred': LABEL_NAMES[cp], 'obf_pred': LABEL_NAMES[op]}
    for i, (cp, op, tl) in enumerate(zip(preds_clean_4a, p_obf, test_labels))
    if cp == tl and op != tl
]
print(f'\nPosts flipped: {len(failures)} / {len(test_labels)} '
      f'({100*len(failures)/len(test_labels):.1f}%)\n')
for ex in random.sample(failures, min(5, len(failures))):
    print(f"  True: {ex['true']:<12} Clean→{ex['clean_pred']:<10} Obf→{ex['obf_pred']}")
    print(f"    {ex['text']}")
    print(f"    → {ex['obf']}\n")

---
## Task 4b — Improved Training (Balanced Class Weights)

One improvement over the baseline:

1. **Class weights** — the loss penalises errors on minority classes (`hate`, `offensive`) more than `normal`, directly reducing false negatives on the classes that matter most

In [ ]:
# ── Class weights ─────────────────────────────────────────
labels_array         = train_df["label"].values
class_weights        = compute_class_weight("balanced", classes=np.array([0,1,2]), y=labels_array)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion_4b         = nn.CrossEntropyLoss(weight=class_weights_tensor)

print("Class weights:")
for i, w in enumerate(class_weights):
    print(f"  {LABEL_NAMES[i]:<12} → {w:.4f}")

# ── Datasets ─────────────────────────────────────────────
train_ds_4b = HateDataset(train_df["clean_text"].tolist(), train_df["label"].tolist(), tokenizer)
val_ds_4b   = HateDataset(val_df["clean_text"].tolist(),   val_df["label"].tolist(),   tokenizer)
test_ds_4b  = HateDataset(test_df["clean_text"].tolist(),  test_df["label"].tolist(),  tokenizer)

train_loader_4b = DataLoader(train_ds_4b, batch_size=BATCH_SIZE, shuffle=True)
val_loader_4b   = DataLoader(val_ds_4b,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_4b  = DataLoader(test_ds_4b,  batch_size=BATCH_SIZE, shuffle=False)

# Fresh model instance
model_4b = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
print(f"✓ Improved datasets and model ready")

In [ ]:
# ── Train ─────────────────────────────────────────────────
best_state_4b, history_4b, stopped_4b = train_model(
    model_4b, train_loader_4b, val_loader_4b,
    epochs=MAX_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
    patience=PATIENCE, criterion=criterion_4b,
    label="— HateBERT improved"
)
plot_curves(history_4b, stopped_4b, title="— HateBERT Improved",
            save_path="training_curves_4b.png")

In [ ]:
# ── Restore best model & evaluate on test set ─────────────
model_4b.load_state_dict(best_state_4b)
_, _, preds_4b_test, labels_4b_test = evaluate(model_4b, test_loader_4b)

print("── Task 4b — Test Set Results ──")
print(classification_report(labels_4b_test, preds_4b_test,
                             target_names=["hate","offensive","normal"], digits=4))
plot_cm(labels_4b_test, preds_4b_test,
        title="— HateBERT Improved", cmap="Greens",
        save_path="cm_4b.png")

# Save
model_4b.save_pretrained(MODEL_IMP_SAVE)
tokenizer.save_pretrained(MODEL_IMP_SAVE)
print(f"✓ Saved to ./{MODEL_IMP_SAVE}/")

### Task 4b — Robustness (Targeted Obfuscation)

In [ ]:
# ── Improved robustness: hate-lexicon words only ─────────
print('── Improved robustness ──')
results_4b, obf_cache_4b, preds_clean_4b = robustness_eval(
    model_4b, tokenizer, test_texts, test_labels,
    obf_fns={'targeted': targeted_obfuscation})

clean_f1 = results_4b['clean']
obf_f1   = results_4b['targeted']
print(f'  clean    → F1: {clean_f1:.4f}')
print(f'  targeted → F1: {obf_f1:.4f}  (drop: {clean_f1-obf_f1:+.4f})')

p_obf, _ = get_predictions(model_4b, tokenizer, obf_cache_4b['targeted'], test_labels)
f1s_c = f1_score(test_labels, preds_clean_4b, average=None)
f1s_o = f1_score(test_labels, p_obf, average=None)
header = f"{'Condition':<12} {'hate':>8} {'offensive':>10} {'normal':>8} {'macro':>8}"
print('\n' + header); print('─'*len(header))
print(f'  clean        {f1s_c[0]:8.4f} {f1s_c[1]:10.4f} {f1s_c[2]:8.4f} {clean_f1:8.4f}')
print(f'  targeted     {f1s_o[0]:8.4f} {f1s_o[1]:10.4f} {f1s_o[2]:8.4f} {obf_f1:8.4f}')

print('\n── Δ Summary (baseline vs improved) ──')
print(f"  {'Model':<22} {'Clean F1':>10} {'Obf F1':>10} {'Drop':>8}")
print('─'*55)
for lbl, res in [('Baseline (4a)', results_4a), ('Improved (4b)', results_4b)]:
    c = res['clean']; o = res['targeted']
    print(f'  {lbl:<22} {c:10.4f} {o:10.4f} {c-o:+8.4f}')

---
## Task 4c — Targeted Augmentation

Adds targeted augmentation on top of balanced class weights: hate and offensive
posts are obfuscated with `targeted_obfuscation` (hate-lexicon words only) at
probability `AUG_RATE` during each training epoch.

For HateBERT the expected benefit is smaller than for BiLSTM: WordPiece already
decomposes unknown tokens into subword pieces, providing partial robustness without
any augmentation. Task 4c tests whether explicit exposure to obfuscated hate words
during fine-tuning adds further improvement.

In [ ]:
# ── Datasets with targeted augmentation ──────────────────
train_ds_4c = HateDataset(
    train_df['clean_text'].tolist(), train_df['label'].tolist(),
    tokenizer, augment_fn=targeted_obfuscation, aug_rate=AUG_RATE
)
val_ds_4c  = HateDataset(val_df['clean_text'].tolist(),  val_df['label'].tolist(),  tokenizer)
test_ds_4c = HateDataset(test_df['clean_text'].tolist(), test_df['label'].tolist(), tokenizer)

train_loader_4c = DataLoader(train_ds_4c, batch_size=BATCH_SIZE, shuffle=True)
val_loader_4c   = DataLoader(val_ds_4c,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_4c  = DataLoader(test_ds_4c,  batch_size=BATCH_SIZE, shuffle=False)

model_4c = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
print(f'✓ Task 4c ready  aug_rate={AUG_RATE} | class weights reused from 4b')

In [ ]:
# ── Train ─────────────────────────────────────────────────
best_state_4c, history_4c, stopped_4c = train_model(
    model_4c, train_loader_4c, val_loader_4c,
    epochs=MAX_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
    patience=PATIENCE, criterion=criterion_4b,
    label='— targeted augmentation'
)
plot_curves(history_4c, stopped_4c, title='— Targeted Augmentation',
            save_path='training_curves_4c.png')

  Ep 3 | Step 200/962 | Loss 0.4735


In [ ]:
# ── Restore best model & evaluate on test set ─────────────
model_4c.load_state_dict(best_state_4c)
_, _, preds_4c_test, labels_4c_test = evaluate(model_4c, test_loader_4c)

print('── Task 4c — Test Results ──')
print(classification_report(labels_4c_test, preds_4c_test,
                             target_names=['hate','offensive','normal'], digits=4))
plot_cm(labels_4c_test, preds_4c_test, title='— Targeted Aug.', cmap='Purples', save_path='cm_4c.png')

In [ ]:
# ── Task 4c robustness + 3-way comparison ────────────────
print('── Targeted augmentation robustness ──')
results_4c, obf_cache_4c, preds_clean_4c = robustness_eval(
    model_4c, tokenizer, test_texts, test_labels,
    obf_fns={'targeted': targeted_obfuscation})

clean_f1 = results_4c['clean']
obf_f1   = results_4c['targeted']
print(f'  clean    → F1: {clean_f1:.4f}')
print(f'  targeted → F1: {obf_f1:.4f}  (drop: {clean_f1-obf_f1:+.4f})')

print('\n── Targeted obfuscation eval (hate lexicon words only) ──')
print(f"  {'Model':<24} {'Clean F1':>10} {'Obf F1':>10} {'Drop':>8}")
print('─'*57)
for lbl, res in [
    ('Baseline (4a)',      results_4a),
    ('Improved (4b)',      results_4b),
    ('Targeted aug. (4c)', results_4c),
]:
    c = res['clean']; o = res['targeted']
    print(f'  {lbl:<24} {c:10.4f} {o:10.4f} {c-o:+8.4f}')

---
## 💾 Save Models to Google Drive

---
## Threshold Tuning — Hate Class Recall Analysis

The baseline model uses argmax (highest probability wins). By lowering the decision threshold for the `hate` class, we can increase recall at the cost of precision. This is desirable in content moderation, where missing a hateful post (false negative) is worse than flagging a normal one (false positive).

In [ ]:
# ── Threshold sweep on test set (baseline model) ─────────
from sklearn.metrics import precision_score, recall_score

test_texts  = test_df["clean_text"].tolist()
test_labels = test_df["label"].tolist()

_, all_probs = get_predictions(model_4a, tokenizer, test_texts, test_labels)

thresholds = np.arange(0.15, 0.55, 0.025)
results_th = []

for th in thresholds:
    preds_th = []
    for prob in all_probs:
        if prob[0] >= th:
            preds_th.append(0)
        else:
            preds_th.append(prob.argmax())

    prec = precision_score(test_labels, preds_th, labels=[0], average=None)[0]
    rec  = recall_score(test_labels, preds_th, labels=[0], average=None)[0]
    f1_h = f1_score(test_labels, preds_th, labels=[0], average=None)[0]
    f1_m = f1_score(test_labels, preds_th, average="macro")
    results_th.append({"threshold": th, "hate_precision": prec,
                        "hate_recall": rec, "hate_f1": f1_h, "macro_f1": f1_m})

df_th = pd.DataFrame(results_th)
print(df_th.to_string(index=False, float_format="%.3f"))

In [ ]:
# ── Precision-Recall tradeoff plot ────────────────────────
fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(df_th["threshold"], df_th["hate_recall"], "o-", color="#e63946",
         linewidth=2, label="Hate Recall")
ax1.plot(df_th["threshold"], df_th["hate_precision"], "o-", color="#457b9d",
         linewidth=2, label="Hate Precision")
ax1.plot(df_th["threshold"], df_th["hate_f1"], "o--", color="#f4a261",
         linewidth=2, label="Hate F1")

ax1.axvline(1/3, color="gray", linestyle=":", alpha=0.7, label="Default (argmax ≈ 0.33)")

ax1.set_xlabel("Hate Class Threshold", fontweight="bold")
ax1.set_ylabel("Score", fontweight="bold")
ax1.set_title("Threshold Tuning — Hate Class Precision vs Recall", fontweight="bold")
ax1.legend(loc="center right")
ax1.set_ylim(0.4, 1.0)
ax1.spines[["top", "right"]].set_visible(False)
ax1.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("threshold_tuning.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved threshold_tuning.png")

In [ ]:
# ── Compare default vs tuned threshold ────────────────────
default_row = df_th.iloc[(df_th["threshold"] - 1/3).abs().argsort().iloc[0]]
best_recall_row = df_th[df_th["hate_recall"] >= 0.90].iloc[-1] if (df_th["hate_recall"] >= 0.90).any() else df_th.iloc[df_th["hate_recall"].argmax()]

print("── Default (argmax) ──")
print(f"   Threshold     : {default_row['threshold']:.3f}")
print(f"   Hate Precision: {default_row['hate_precision']:.3f}")
print(f"   Hate Recall   : {default_row['hate_recall']:.3f}")
print(f"   Hate F1       : {default_row['hate_f1']:.3f}")
print(f"   Macro F1      : {default_row['macro_f1']:.3f}")

print(f"\n── Tuned (recall ≥ 0.90) ──")
print(f"   Threshold     : {best_recall_row['threshold']:.3f}")
print(f"   Hate Precision: {best_recall_row['hate_precision']:.3f}")
print(f"   Hate Recall   : {best_recall_row['hate_recall']:.3f}")
print(f"   Hate F1       : {best_recall_row['hate_f1']:.3f}")
print(f"   Macro F1      : {best_recall_row['macro_f1']:.3f}")

delta_rec  = best_recall_row['hate_recall'] - default_row['hate_recall']
delta_prec = best_recall_row['hate_precision'] - default_row['hate_precision']
print(f"\n── Tradeoff ──")
print(f"   Recall gain   : +{delta_rec:.3f}")
print(f"   Precision cost: {delta_prec:.3f}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import shutil, os

os.makedirs(DRIVE_DIR, exist_ok=True)

for src, dst_name in [(MODEL_SAVE, "best_model_baseline"),
                       (MODEL_IMP_SAVE, "best_model_improved")]:
    dst = os.path.join(DRIVE_DIR, dst_name)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"✓ Saved {src} → {dst}")